# EfficientNet-B0 Pretrained — NWPU-RESISC45
**Model 4** | ImageNet weights · ~5.3M params · Compound scaling · CE loss · Adam · Augment

## Imports

In [2]:
import time
import os
from typing import Dict, List

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchinfo import summary
from torchvision import datasets, transforms
import torchvision.models as models

## Device Check

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

True
0
NVIDIA T1200 Laptop GPU
Using device: cuda


## Dataset Preparation

### 1. Transforms

EfficientNet-B0 expects **224×224** input with standard ImageNet normalization.
We use the same augmentation pipeline as the other models for a fair comparison.

In [4]:
def get_resisc45_transforms(img_size=224, augment=True):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]) if augment else transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    val_tf = transforms.Compose([
        transforms.Resize(int(img_size * 1.14)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    return train_tf, val_tf

### 2. DataLoader Setup

In [5]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

    def __len__(self):
        return len(self.subset)

In [6]:
def get_dataloaders(data_root, img_size=224, batch_size=24):
    # Batch size 24 — same as Models 1 & 3
    # EfficientNet uses more memory than MobileNetV2 due to larger feature maps

    train_tf, val_tf = get_resisc45_transforms(img_size)

    base_dataset = datasets.ImageFolder(root=data_root)

    n_total = len(base_dataset)
    n_train = int(0.7 * n_total)
    n_val   = int(0.15 * n_total)
    n_test  = n_total - n_train - n_val

    # Same seed as all other models — identical train/val/test splits
    train_ds, val_ds, test_ds = random_split(
        base_dataset,
        [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

    train_ds = TransformSubset(train_ds, train_tf)
    val_ds   = TransformSubset(val_ds, val_tf)
    test_ds  = TransformSubset(test_ds, val_tf)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True
    )

    return train_loader, val_loader, test_loader

## Architecture — EfficientNet-B0 Pretrained

EfficientNet uses **compound scaling** — simultaneously scaling depth, width, and resolution with a fixed ratio.
B0 is the baseline variant with ~5.3M parameters, sitting between MobileNetV2 (~3.4M) and ResNet18 (~11M).

Key architectural features:
- **MBConv blocks** (Mobile Inverted Bottleneck) with Squeeze-and-Excitation attention
- **Swish activation** (smoother than ReLU, better gradient flow)
- **Stochastic depth** (drop-path regularization built-in)

We load ImageNet pretrained weights and replace the final classifier for 45 classes.

In [7]:
def build_efficientnet_b0(num_classes: int = 45, dropout_p: float = 0.3) -> nn.Module:
    """
    Load torchvision EfficientNet-B0 with ImageNet pretrained weights.
    Replace the classifier head to output `num_classes` logits.

    EfficientNet-B0 classifier: nn.Sequential(Dropout, Linear(1280 -> 1000))
    We replace it with:         nn.Sequential(Dropout, Linear(1280 -> num_classes))
    """
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    # Replace classifier head — in_features is 1280 for B0
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_p, inplace=True),
        nn.Linear(in_features, num_classes)
    )

    return model


def count_params(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters  : {total:,}")
    print(f"Trainable params  : {trainable:,}")
    print(f"Model size (FP32) : {total * 4 / 1024**2:.2f} MB")

## Training Functions

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(
            device,
            non_blocking=True,
            dtype=torch.float32
        )

        labels = labels.to(
            device,
            non_blocking=True
        )
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return {
        "loss": total_loss / total,
        "acc": correct / total
    }

### Evaluate

In [9]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

    return {"loss": total_loss / total, "acc": correct / total}

### Full Training Loop

EfficientNet trains end-to-end from pretrained weights in a single phase.
Uses CE loss + Adam + cosine LR schedule + label smoothing + early stopping.

In [ ]:
def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 25,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    save_path: str = "mobilenetv2_best.pth",
    device_str: str = "auto",
):


    device = (
        torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if device_str == "auto"
        else torch.device(device_str)
    )

    print(f"Training on: {device}")

    model = model.to(device)


    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )

    best_val_acc = 0.0
    patience_ctr = 0
    patience = 5

    history = []

    for epoch in range(1, num_epochs + 1):

        t0 = time.time()

        train_stats = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_stats = evaluate(
            model,
            val_loader,
            criterion,
            device
        )


        scheduler.step()
        elapsed = time.time() - t0
        lr_now = optimizer.param_groups[0]['lr']


        print(
            f"Epoch {epoch:3d}/{num_epochs}  "
            f"Train Loss: {train_stats['loss']:.4f} | "
            f"Train Acc: {train_stats['acc']:.4f} | "
            f"Val Loss: {val_stats['loss']:.4f} | "
            f"Val Acc: {val_stats['acc']:.4f} | "
            f"LR: {lr_now:.2e} | "
            f"{elapsed:.1f}s"
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_stats["loss"],
            "train_acc": train_stats["acc"],
            "val_loss": val_stats["loss"],
            "val_acc": val_stats["acc"],
            "lr": lr_now,
        })


        if val_stats['acc'] > best_val_acc:

            best_val_acc = val_stats['acc']

            patience_ctr = 0

            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "val_acc": best_val_acc,
            }, save_path)

            print(f"  ✅ New best saved: {best_val_acc:.4f}")

        else:
            patience_ctr += 1

            if patience_ctr >= patience:

                print(
                    f"\n Early stopping at epoch {epoch} "
                    f"(no improvement for {patience} epochs)"
                )

                break

    print(f"\nTraining complete. Best val acc: {best_val_acc:.4f}")

    return history

## Run

In [ ]:
# -------------------------------
# CONFIG
# -------------------------------
data_path  = r"D:\DELL\Documents\deeplearning_proj\NWPU-RESISC45"   # same as all other models
MODEL_PATH = "efficientnet_b0_best.pth"
TRAIN      = True   # Set False to skip training and load checkpoint

# -------------------------------
# MODEL
# -------------------------------
model = build_efficientnet_b0(num_classes=45, dropout_p=0.3)
print("=== EfficientNet-B0 — Model Summary ===")
count_params(model)

# -------------------------------
# DATA
# -------------------------------
if TRAIN:
    train_loader, val_loader, test_loader = get_dataloaders(data_path)
else:
    _, _, test_loader = get_dataloaders(data_path)

# Sanity check
test_batch_img, test_batch_lbl = next(iter(test_loader))
test_batch_img = test_batch_img.to(device)
print(f"Test batch on device: {test_batch_img.device}")
print(f"Batch shape: {test_batch_img.shape}")

# Quick forward pass check
model_tmp = model.to(device)
with torch.no_grad():
    out = model_tmp(test_batch_img)
print(f"Forward pass output shape: {out.shape}   (expected: [{test_batch_img.shape[0]}, 45])")

# -------------------------------
# TRAIN OR LOAD
# -------------------------------
if TRAIN:
    print("\nStarting training...\n")
    history = train(
        model,
        train_loader,
        val_loader,
        num_epochs=40,
        lr=1e-4,
        weight_decay=1e-4,
        patience=5,
        save_path=MODEL_PATH
    )
    model.eval()
else:
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Checkpoint not found: {MODEL_PATH}")

    print("\nLoading trained model...\n")
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    model.eval()

print("\nModel is ready ✔️")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\DELL/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:07<00:00, 2.86MB/s]


=== EfficientNet-B0 — Model Summary ===
Total parameters  : 4,065,193
Trainable params  : 4,065,193
Model size (FP32) : 15.51 MB
Test batch on device: cuda:0
Batch shape: torch.Size([24, 3, 224, 224])
Forward pass output shape: torch.Size([24, 45])   (expected: [24, 45])

Starting training...

Training on: cuda
Epoch   1/40  Train Loss: 1.7399 | Train Acc: 0.7043 | Val Loss: 1.0099 | Val Acc: 0.9043 | LR: 9.98e-05 | 706.3s
-- New best saved: 0.9043
Epoch   2/40  Train Loss: 1.0352 | Train Acc: 0.8956 | Val Loss: 0.9117 | Val Acc: 0.9350 | LR: 9.94e-05 | 367.7s
-- New best saved: 0.9350
Epoch   3/40  Train Loss: 0.9440 | Train Acc: 0.9250 | Val Loss: 0.8783 | Val Acc: 0.9437 | LR: 9.86e-05 | 362.2s
-- New best saved: 0.9437
Epoch   4/40  Train Loss: 0.8942 | Train Acc: 0.9448 | Val Loss: 0.8530 | Val Acc: 0.9537 | LR: 9.76e-05 | 360.0s
-- New best saved: 0.9537
Epoch   5/40  Train Loss: 0.8638 | Train Acc: 0.9528 | Val Loss: 0.8442 | Val Acc: 0.9553 | LR: 9.62e-05 | 363.4s
-- New best s

## Evaluation (Model Metrics)

In [11]:
def print_model_metrics(model: nn.Module, img_size: int = 224):
    device = next(model.parameters()).device

    stats = summary(
        model,
        input_size=(1, 3, img_size, img_size),
        device=device,
        verbose=0,
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    size_mb = total_params * 4 / (1024 ** 2)

    print(f"\n{'='*50}")
    print(f"  Total parameters  : {total_params:,}")
    print(f"  Trainable params  : {trainable:,}")
    print(f"  Model size (FP32) : {size_mb:.2f} MB")
    print(f"  Total FLOPs       : {stats.total_mult_adds / 1e9:.2f} GFLOPs")
    print(f"{'='*50}\n")

    print("Comparison vs other models:")
    print("  - ResNet18 scratch    : ~11M params | baseline accuracy")
    print("  - ResNet18 fine-tuned : ~11M params | best accuracy expected")
    print("  - MobileNetV2         : ~3.4M params | fastest / smallest")
    print("  - EfficientNet-B0     : ~5.3M params | best accuracy/efficiency trade-off")
    print(f"\nAfter INT8 quantization:")
    print(f"  - Estimated size: ~{size_mb/4:.2f} MB")
    print(f"  - ~2–3× faster CPU inference")
    print(f"  - Minimal accuracy drop (typically <1%)\n")

In [12]:
# Load best checkpoint for final evaluation
checkpoint = torch.load(MODEL_PATH, weights_only=False)
model = build_efficientnet_b0(num_classes=45)
model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

criterion = nn.CrossEntropyLoss()

# -------------------------------
# TEST EVALUATION
# -------------------------------
test_stats = evaluate(model, test_loader, criterion, device)

print("\n=== TEST RESULTS ===")
print(f"Test Accuracy: {test_stats['acc']:.4f}")
print(f"Test Loss    : {test_stats['loss']:.4f}")

# -------------------------------
# MODEL METRICS
# -------------------------------
print("\n=== MODEL METRICS (FP32) ===")
print_model_metrics(model)


=== TEST RESULTS ===
Test Accuracy: 0.9594
Test Loss    : 0.2623

=== MODEL METRICS (FP32) ===

  Total parameters  : 4,065,193
  Trainable params  : 4,065,193
  Model size (FP32) : 15.51 MB
  Total FLOPs       : 0.38 GFLOPs

Comparison vs other models:
  - ResNet18 scratch    : ~11M params | baseline accuracy
  - ResNet18 fine-tuned : ~11M params | best accuracy expected
  - MobileNetV2         : ~3.4M params | fastest / smallest
  - EfficientNet-B0     : ~5.3M params | best accuracy/efficiency trade-off

After INT8 quantization:
  - Estimated size: ~3.88 MB
  - ~2–3× faster CPU inference
  - Minimal accuracy drop (typically <1%)



## Quantize + ONNX Export

In [13]:
def quantize_dynamic(model: nn.Module, save_path: str = "efficientnet_b0_dynamic_int8.pth") -> nn.Module:
    """
    Dynamic INT8 quantization — quantizes Linear layers.
    No calibration needed, fastest to apply.
    """
    model_cpu = model.cpu().eval()

    quantized = torch.quantization.quantize_dynamic(
        model_cpu,
        qconfig_spec={nn.Linear},
        dtype=torch.qint8,
    )

    torch.save(quantized.state_dict(), save_path)
    print(f"Dynamic-quantized model saved -> {save_path}")
    return quantized


def quantize_static(
    model: nn.Module,
    calibration_loader: DataLoader,
    save_path: str = "efficientnet_b0_static_int8.pth",
) -> nn.Module:
    """
    Static INT8 PTQ (as per diagram) — FP32 → INT8, ~4× size reduction.
    Calibrates both Conv and Linear layers.
    """
    model_cpu = model.cpu().eval()
    model_cpu.qconfig = torch.quantization.get_default_qconfig("x86")

    torch.quantization.prepare(model_cpu, inplace=True)

    print("Calibrating for static quantization...")
    with torch.no_grad():
        for i, (images, _) in enumerate(calibration_loader):
            model_cpu(images)
            if i >= 10:
                break

    torch.quantization.convert(model_cpu, inplace=True)
    torch.save(model_cpu.state_dict(), save_path)
    print(f"Static-quantized model saved -> {save_path}")
    return model_cpu


def export_to_onnx(model, save_path="efficientnet_b0.onnx", img_size=224, opset=17):
    model = model.cpu().eval()
    dummy_input = torch.randn(1, 3, img_size, img_size)

    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        opset_version=opset,
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch_size"}, "logits": {0: "batch_size"}},
        do_constant_folding=True
    )
    print("ONNX exported:", save_path)


def validate_onnx(onnx_path: str, img_size: int = 224, num_classes: int = 45):
    import onnxruntime as ort
    import numpy as np

    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    dummy = np.random.randn(1, 3, img_size, img_size).astype(np.float32)
    outputs = session.run(None, {"image": dummy})

    assert outputs[0].shape == (1, num_classes), \
        f"Unexpected output shape: {outputs[0].shape}"

    print(f"ONNX validation passed - output shape: {outputs[0].shape}")
    return outputs[0]

In [14]:
# -------------------------------
# QUANTIZATION (PTQ)
# -------------------------------
print("\nApplying dynamic quantization...\n")
q_model = quantize_dynamic(model)
print("\n=== MODEL METRICS (INT8) ===")
print_model_metrics(q_model)


Applying dynamic quantization...

Dynamic-quantized model saved -> efficientnet_b0_dynamic_int8.pth

=== MODEL METRICS (INT8) ===

  Total parameters  : 4,007,548
  Trainable params  : 4,007,548
  Model size (FP32) : 15.29 MB
  Total FLOPs       : 0.38 GFLOPs

Comparison vs other models:
  - ResNet18 scratch    : ~11M params | baseline accuracy
  - ResNet18 fine-tuned : ~11M params | best accuracy expected
  - MobileNetV2         : ~3.4M params | fastest / smallest
  - EfficientNet-B0     : ~5.3M params | best accuracy/efficiency trade-off

After INT8 quantization:
  - Estimated size: ~3.82 MB
  - ~2–3× faster CPU inference
  - Minimal accuracy drop (typically <1%)



In [15]:
# -------------------------------
# ONNX EXPORT + VALIDATION
# -------------------------------
print("\nExporting to ONNX...\n")
export_to_onnx(model, "efficientnet_b0.onnx")
validate_onnx("efficientnet_b0.onnx")

print("\nAll done ✔️")


Exporting to ONNX...

ONNX exported: efficientnet_b0.onnx
ONNX validation passed - output shape: (1, 45)

All done ✔️
